# Практическая работа № 6. Обработка текстовых данных с использованием Spark MLlib

Выполнил: Савинский Владимир Олегович, ИКБО-40-23,  03.11.2026

**Цель:** построить воспроизводимый процесс анализа русскоязычного корпуса: чтение и контроль данных, подготовка текстов, частотный анализ, разреженные представления BoW и TF–IDF, классификация и генерация с помощью цепей Маркова.

**Задача:** определить спортивную тематику сообщества по содержимому его публикаций. Целевая переменная - `category`, признаки извлекаются только из `text`. Поле `oid` используется для аудита, но не передаётся классификатору.

**Данные:** [Russian Social Media Text Classification](https://www.kaggle.com/datasets/mikhailma/russian-social-media-text-classification/data).

| Поле | Назначение |
|---|---|
| `oid` | Идентификатор записи в исходном наборе |
| `category` | Одна из 13 спортивных тематик |
| `text` | Русскоязычный текст, отдельная публикация либо текстовый объект в формате источника |

**Адаптация методички.** За основу взята полная методичка на 43 страницах. В ней учебный пример посвящён отзывам, самостоятельная работа - авторству литературных произведений. По условию данной работы они заменены спортивными публикациями: автор → тематика, произведение → исходный текстовый объект, фрагмент произведения → часть длинной публикации. Сохранены сравнение моделей, два уровня оценки, анализ ошибок, сохранение решения и генераторы двух порядков. Это адаптация содержания, а не выполнение задания на Literature.zip.

**Оформление:** как в ноутбуке-примере - постановка задачи, нумерованные разделы, пояснение перед кодом, выводы после этапов, итог и ограничения. Выводы с числами создаются кодом после расчётов. В поставляемом ноутбуке нет вымышленных результатов обучения.

**Статус:** ноутбук выполнен на полном размеченном train.csv; сохранены выводы всех ячеек. Окружение проверенного запуска: Python 3.12, Spark 3.5.3, Java 17. Численные результаты относятся к зафиксированному разбиению и параметрам.


## 1. Окружение, параметры и загрузка данных

Расчёты корпуса и обучение выполняются в Spark. В Python передаются небольшие агрегированные таблицы, ограниченные примеры ошибок и таблица переходов с контролем размера.

Для воспроизводимости рекомендуется Python 3.10–3.11, Java 17 и PySpark 3.5.3. В терминале окружения Jupyter:

```bash
python -m pip install pyspark==3.5.3 pandas numpy matplotlib kagglehub wordcloud jupyter
```

На Windows Java должна быть доступна через `JAVA_HOME`. Если в вашем окружении необходим Hadoop, задайте `HADOOP_HOME` и добавьте его `bin` в PATH перед запуском Jupyter, как в ноутбуке-примере.

**Запуск:** положите `train.csv` рядом с ноутбуком и выберите «Restart Kernel and Run All». Если файла нет, включена автоматическая загрузка через kagglehub; она требует доступа к Kaggle и иногда авторизации. Альтернатива - переменная окружения `SPORTS_CSV` с полным путём к файлу. Результаты сохраняются в `results_pr6`, промежуточные данные - в `data_pr6`. Полный запуск обучает шесть моделей и может занять значительное время.


In [ ]:
import os, sys, json, csv, math, random, re, platform, hashlib
from pathlib import Path
from collections import defaultdict, Counter
from time import perf_counter

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ.setdefault('SPARK_LOCAL_IP', '127.0.0.1')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import pyspark
from pyspark.sql import SparkSession, functions as F, types as T, Window
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import (SQLTransformer, RegexTokenizer, StopWordsRemover,
    CountVectorizer, CountVectorizerModel, IDF, Normalizer, NGram, HashingTF)
from pyspark.ml.classification import LogisticRegression, NaiveBayes
from pyspark.ml.functions import vector_to_array

SEED = 42
ROOT = Path.cwd().resolve()
WORK, OUT = ROOT / 'data_pr6', ROOT / 'results_pr6'
WORK.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
(OUT / 'figures').mkdir(exist_ok=True)
AUTO_DOWNLOAD = True
VOCAB_SIZE, MIN_DF, REG_PARAM = 20000, 3.0, 0.001
FRAGMENT_WORDS = 300
MAX_TRAIN_FRAGMENTS = 50
MARKOV_DOCS, MAX_EDGES = 300, 200000
RUN_STOP_ABLATION = True

CATEGORIES = ['athletics', 'autosport', 'basketball', 'boardgames', 'esport',
    'extreme', 'football', 'hockey', 'martial_arts', 'motosport', 'tennis',
    'volleyball', 'winter_sport']
LABEL_NAMES = dict(enumerate(CATEGORIES))
K = len(CATEGORIES)
label_map = spark_map = None

spark = (SparkSession.builder.appName('Practice6_SportsText')
    .master(os.environ.get('SPARK_MASTER', 'local[2]'))
    .config('spark.driver.memory', os.environ.get('SPARK_DRIVER_MEMORY', '4g'))
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.session.timeZone', 'UTC').getOrCreate())
spark.sparkContext.setLogLevel('WARN')
sc = spark.sparkContext
plt.rcParams.update({'figure.figsize': (9, 4.5), 'font.family': 'DejaVu Sans'})

def conclusion(text):
    display(Markdown('**Вывод:** ' + text))

def savefig(name):
    plt.tight_layout()
    plt.savefig(OUT / 'figures' / name, dpi=150, bbox_inches='tight')
    plt.show()

print('Python:', platform.python_version(), '| Spark:', spark.version)
print('Каталог:', ROOT, '| master:', sc.master)


In [ ]:
if os.environ.get('SPORTS_CSV'):
    CSV_PATH = Path(os.environ['SPORTS_CSV']).expanduser().resolve()
else:
    CSV_PATH = ROOT / 'train.csv'
    if not CSV_PATH.is_file():
        if not AUTO_DOWNLOAD:
            raise FileNotFoundError('Поместите train.csv рядом с ноутбуком или задайте SPORTS_CSV')
        import kagglehub
        try:
            downloaded = Path(kagglehub.dataset_download(
                'mikhailma/russian-social-media-text-classification'))
        except Exception as exc:
            raise RuntimeError('Не удалось скачать Kaggle. Скачайте train.csv вручную и задайте SPORTS_CSV.') from exc
        candidates = sorted(downloaded.rglob('train.csv'))
        if len(candidates) != 1:
            raise ValueError(f'Ожидался один train.csv, найдены: {candidates}')
        CSV_PATH = candidates[0]
if not CSV_PATH.is_file():
    raise FileNotFoundError(CSV_PATH)

with CSV_PATH.open(encoding='utf-8-sig', newline='') as f:
    sample = f.read(65536)
    try:
        DELIMITER = csv.Sniffer().sniff(sample, delimiters=',;\t').delimiter
    except csv.Error:
        DELIMITER = ','
    f.seek(0)
    HEADER = next(csv.reader(f, delimiter=DELIMITER))
assert len(HEADER) == len(set(HEADER)), 'Повторяющиеся имена столбцов'
assert {'oid', 'category', 'text'} <= set(HEADER), f'Неожиданные столбцы: {HEADER}'

# Схема в порядке заголовка; oid сохраняется строкой, чтобы не менять формат ID.
schema = T.StructType([T.StructField(c, T.StringType(), True) for c in HEADER])
raw = (spark.read.schema(schema).option('header', True)
    .option('encoding', 'UTF-8').option('sep', DELIMITER)
    .option('multiLine', True).option('quote', '"').option('escape', '"')
    .option('mode', 'FAILFAST').option('enforceSchema', False)
    .csv(CSV_PATH.as_posix()).select('oid', 'category', 'text'))
N_RAW = raw.count()
assert N_RAW > 0, 'Пустой CSV'
raw.printSchema()
raw.show(3, truncate=120)
print('Разделитель:', repr(DELIMITER), '| размер МБ:', round(CSV_PATH.stat().st_size / 2**20, 2))
print('Строк:', N_RAW, '| партиций при чтении:', raw.rdd.getNumPartitions())

# CSV с многострочными полями может читаться одним заданием на файл.
# Parquet далее позволяет распределить последующие стадии.
raw.repartition(8).write.mode('overwrite').parquet((WORK / 'raw_parquet').as_posix())
raw = spark.read.parquet((WORK / 'raw_parquet').as_posix()).cache()
conclusion(f'Прочитано **{N_RAW:,}** строк с явной строковой схемой. '
           'Переводы строк внутри кавычек учитываются; последующая обработка использует Parquet.')


## 2. Качество корпуса и целевые классы

Проверяются пропуски, пустые тексты, дубликаты и противоречивые метки. Пробел в `martial arts` приводится к `_`; получаем единое написание `martial_arts`. Отсутствующие и неизвестные метки не угадываются: строки без меток исключаются с отчётом, неизвестные категории останавливают работу.

Полные повторы `(oid, category, text)` удаляются. Разные записи с одинаковыми нормализованными токенами сохраняются, но попадут в одну часть выборки. Непустой текст, не распознанный токенизатором, сохраняется как отдельный диагностический случай.


In [ ]:
nulls = raw.agg(*[F.sum(F.col(c).isNull().cast('long')).alias(c) for c in raw.columns]).first().asDict()
empty_text_n = raw.filter(F.col('text').isNull() | (F.length(F.trim('text')) == 0)).count()
missing_label_n = raw.filter(F.col('category').isNull() | (F.length(F.trim('category')) == 0)).count()
print('Пропуски:', nulls, '| пустой текст:', empty_text_n, '| без метки:', missing_label_n)

base = (raw.filter(F.col('category').isNotNull() & (F.length(F.trim('category')) > 0))
    .withColumn('category', F.regexp_replace(F.lower(F.trim('category')), r'[ -]+', '_'))
    .withColumn('text', F.coalesce('text', F.lit(''))))
unknown = [r.category for r in base.select('category').distinct().collect()
           if r.category not in CATEGORIES]
assert not unknown, f'Неизвестные категории: {unknown}. Сверьте версию датасета.'
N_BEFORE_DEDUP = base.count()
base = base.dropDuplicates(['oid', 'category', 'text'])
N_DOCS = base.count()
DUPLICATES = N_BEFORE_DEDUP - N_DOCS
mapping = F.create_map(*[v for i, name in LABEL_NAMES.items()
                        for v in (F.lit(name), F.lit(float(i)))])
base = (base.withColumn('label', mapping[F.col('category')])
    .withColumn('row_id', F.sha2(F.to_json(F.struct('oid', 'category', 'text')), 256)))
assert base.select('row_id').distinct().count() == N_DOCS
class_counts = base.groupBy('category').count().orderBy(F.desc('count'), 'category').toPandas()
display(class_counts)
class_counts.to_csv(OUT / 'corpus_classes.csv', index=False)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(class_counts.category[::-1], class_counts['count'][::-1])
ax.set(xlabel='Число исходных текстов', ylabel='Тематика', title='Состав размеченного корпуса после удаления полных повторов')
savefig('01_corpus_classes.png')
conclusion(f'Удалено **{DUPLICATES:,}** полных повторов; без метки исключено **{missing_label_n:,}** строк. '
    f'Для работы осталось **{N_DOCS:,}** текстов. Самый частый класс - '
    f'**{class_counts.iloc[0].category}**. Поэтому accuracy дополняется macro-F1, '
    'где каждая из 13 тематик получает одинаковый вес.')


## 3. Нормализация, токенизация и разбиение без утечки

Сначала задаём правила, не обучаемые по корпусу. Для классификации переводим текст в нижний регистр, заменяем URL и VK-разметку ссылок пробелом, сохраняя видимый текст ссылки. Токенизатор сохраняет русские и латинские слова, числа, внутрисловные дефисы, `!` и `?`. Они могут описывать соревнования, команды, счёт и эмоциональность публикаций. Различие `е`/`ё` и словоформы сохраняются; автоматическое исправление имён и сленга не применяется.

Идентичные после такой подготовки тексты получают общий `normalized_id`. Одновременно все записи одного `oid` объединяются в группу источника. Если одинаковый непустой токенизированный текст встречается у разных источников, такие источники объединяются в связанную компоненту. Хеш компоненты вместе с seed задаёт разбиение примерно **70/15/15**. Все источники связанной компоненты остаются вместе. Для каждого отдельного `oid` проверяется единственность категории; при нарушении оценка по источникам останавливается. Правила не используют целевую метку. Словарь и IDF обучаются позднее исключительно на train.

Фактическая проверка файла показывает 10 записей на `oid`. Поэтому `oid` используется как группа источника независимо от его точного смысла в документации. Все её записи остаются вместе. Итоговые метрики считаются также на уровне `oid`; отождествление этих групп с пабликами требует подтверждения описанием датасета. Непустые нормализованные повторы разных `oid` объединяют группы; пустые последовательности не создают связи, поскольку не содержат лексики.


In [ ]:
TOKEN_PATTERN = r"[а-яёa-z0-9]+(?:[-’'][а-яёa-z0-9]+)*|[!?]+"
# Удаляем адреса, оставляем подпись VK-ссылки вида [club123|название].
CLEAN_SQL = r"""
SELECT *, lower(regexp_replace(
    regexp_replace(coalesce(text, ''), '\\[(?:id|club|public)[0-9]+\\|([^\\]]+)\\]', '$1'),
    'https?://[^\\s]+|www\\.[^\\s]+', ' ')) AS text_norm
FROM __THIS__
"""

def text_stages():
    return [SQLTransformer(statement=CLEAN_SQL),
        RegexTokenizer(inputCol='text_norm', outputCol='tokens', pattern=TOKEN_PATTERN,
            gaps=False, minTokenLength=1, toLowercase=False)]

text_prep = Pipeline(stages=text_stages()).fit(base)  # Только Transformer, без статистик корпуса.
tokenized = text_prep.transform(base)
audit = (tokenized.withColumn('normalized_id', F.sha2(F.to_json('tokens'), 256))
    .withColumn('n_tokens', F.size('tokens')))
groups = audit.groupBy('normalized_id').agg(F.count('*').alias('n'), F.countDistinct('label').alias('n_labels'))
N_GROUPS = groups.count()
CONFLICT_GROUPS = groups.filter('n_labels > 1').count()
# Группы источников: не допускаем oid в нескольких выборках.
audit = audit.withColumn('source_id', F.when(F.length(F.trim('oid')) > 0, F.col('oid')).otherwise(F.col('row_id')))
assert audit.groupBy('source_id').agg(F.countDistinct('label').alias('n')).filter('n > 1').count()==0, 'Один oid имеет разные категории'
# Локально собирается только небольшой граф идентификаторов, не тексты или признаки.
MAX_SOURCE_GRAPH = 100000
source_ids = audit.select('source_id').distinct()
assert source_ids.limit(MAX_SOURCE_GRAPH+1).count() <= MAX_SOURCE_GRAPH, 'Превышен лимит учебного графа источников'
representatives = audit.filter('n_tokens > 0').groupBy('normalized_id').agg(F.min('source_id').alias('representative'))
links = (audit.filter('n_tokens > 0').select('normalized_id','source_id').distinct()
    .join(representatives,'normalized_id').filter(F.col('source_id')!=F.col('representative'))
    .select('source_id','representative').distinct())
assert links.limit(MAX_SOURCE_GRAPH+1).count() <= MAX_SOURCE_GRAPH, 'Слишком много связей повторов'
parent = {r.source_id:r.source_id for r in source_ids.collect()}
def root_of(x):
    while parent[x]!=x:
        parent[x]=parent[parent[x]]
        x=parent[x]
    return x
for link in links.collect():
    a,b=root_of(link.source_id),root_of(link.representative)
    if a!=b:
        parent[max(a,b)]=min(a,b)
source_components = spark.createDataFrame([(x,root_of(x)) for x in sorted(parent)],['source_id','group_id'])
audit = audit.join(F.broadcast(source_components),'source_id')
SOURCE_COUNT = len(parent)
COMPONENT_COUNT = len({root_of(x) for x in parent})
print('Групп oid:',SOURCE_COUNT,'| независимых компонент:',COMPONENT_COUNT)
key = F.sha2(F.concat_ws(':', 'group_id', F.lit(str(SEED))), 256)
bucket = F.conv(F.substring(key, 1, 8), 16, 10).cast('long') % 10000
split_docs = (audit.withColumn('_bucket', bucket)
    .withColumn('split', F.when(F.col('_bucket') < 7000, 'train')
        .when(F.col('_bucket') < 8500, 'valid').otherwise('test'))
    .drop('_bucket').cache())
assert split_docs.filter('n_tokens > 0').groupBy('normalized_id').agg(F.countDistinct('split').alias('n')).filter('n > 1').count()==0
assert split_docs.groupBy('source_id').agg(F.countDistinct('split').alias('n')).filter('n > 1').count()==0
assert split_docs.groupBy('group_id').agg(F.countDistinct('split').alias('n')).filter('n > 1').count() == 0
for name in ['train', 'valid', 'test']:
    present = {int(r.label) for r in split_docs.filter(F.col('split') == name).select('label').distinct().collect()}
    assert present == set(range(K)), f'{name}: не все 13 классов. Нужен пересмотр разбиения до обучения.'

split_table = split_docs.groupBy('split', 'category').count().toPandas().pivot(index='category', columns='split', values='count').fillna(0).astype(int)
display(split_table)
split_table.to_csv(OUT / 'split_classes.csv')
split_docs.select('row_id', 'oid', 'source_id', 'normalized_id', 'group_id', 'category', 'split').write.mode('overwrite').parquet((OUT / 'split_manifest').as_posix())
print('Групп:', N_GROUPS, '| с разными метками:', CONFLICT_GROUPS)
print('Без токенов:', split_docs.filter('n_tokens = 0').count())
split_docs.select('text', 'tokens', 'split').show(3, truncate=100)
conclusion(f'Получено **{N_GROUPS:,}** групп, из них **{CONFLICT_GROUPS:,}** имеют разные метки. '
    'Пересечений групп между train/valid/test нет, все 13 классов представлены в каждой части. '
    'Разбиение зафиксировано и будет одинаковым во всех опытах.')


### Фрагменты длинных публикаций и два уровня оценки

Чтобы сохранить идею проверки «фрагмент / целое произведение» из самостоятельной работы, длинные тексты делим на непересекающиеся части по 300 слов. Короткая публикация остаётся одним фрагментом. Последний неполный фрагмент сохраняется: для коротких постов его отбрасывание привело бы к потере большей части корпуса. Разбиение исходных объектов выполнено **до** нарезки.

Нарезка делается по пробельным словам исходного текста, до удаления пунктуации. Модель получает исходное написание внутри фрагмента. Пустой текст остаётся одним пустым фрагментом. На train используем не более 50 фрагментов одного объекта по устойчивому хешу. На valid и test сохраняем все части. На уровне исходного объекта усредним вероятности его фрагментов. Это оценка по исходным текстовым объектам, а не по пабликам без их подтверждённого идентификатора.


In [ ]:
@F.udf(T.ArrayType(T.StringType(), containsNull=False))
def make_fragments(text):
    words = (text or '').split()
    if not words:
        return ['']
    return [' '.join(words[i:i + FRAGMENT_WORDS]) for i in range(0, len(words), FRAGMENT_WORDS)]

fragments = (split_docs.select('row_id', 'oid', 'group_id', 'category', 'label', 'split',
    F.posexplode(make_fragments('text')).alias('fragment_no', 'text'))
    .withColumn('fragment_id', F.concat_ws(':', 'row_id', F.col('fragment_no').cast('string'))))
rank_window = Window.partitionBy('row_id').orderBy(F.sha2('fragment_id', 256), 'fragment_id')
fragments = (fragments.withColumn('_rank', F.row_number().over(rank_window))
    .filter((F.col('split') != 'train') | (F.col('_rank') <= MAX_TRAIN_FRAGMENTS))
    .drop('_rank').cache())
train = fragments.filter("split = 'train'")
valid = fragments.filter("split = 'valid'")
test = fragments.filter("split = 'test'")
fragment_counts = fragments.groupBy('split', 'category').count().toPandas()
display(fragment_counts.pivot(index='category', columns='split', values='count'))
print('Фрагментов:', fragments.count())
conclusion('Фрагменты каждого исходного объекта остаются в одной выборке. '
    'При итоговой оценке будут приведены результаты по фрагментам и по исходным текстам; '
    'на втором уровне длинный текст получает один голос, как и короткий.')


## 4. Сравнение способов обработки текста

Удаление стоп-слов может уменьшить словарь, но частицы «не», «ни», «нет», «без» меняют смысл. Основные модели сохраняют все токены. Дополнительно сравним стандартный русский стоп-лист Spark и `SAFE_STOP`, в котором сохранены отрицания, «но» и «очень». `DISPLAY_STOP` применяется только к рисункам; он не меняет признаки основной модели.

Для тематической классификации словоформы вроде «хоккей», «хоккея», «хоккеем» могут разделять статистику одного понятия. Лемматизация уменьшила бы это расщепление, но требует отдельной проверки спортивных имён, сокращений и сленга. В данной работе морфология не меняется: это явно зафиксированное решение, а не утверждение о превосходстве исходных форм. В генерации исходные словоформы и служебные слова обязательны для сохранения локальных связей.


In [ ]:
RU_STOP = StopWordsRemover.loadDefaultStopWords('russian')
KEEP_WORDS = {'не', 'ни', 'нет', 'без', 'но', 'очень'}
SAFE_STOP = sorted(set(RU_STOP) - KEEP_WORDS)
DISPLAY_STOP = set(RU_STOP) - {'не', 'нет'}
demo = spark.createDataFrame([
    ('Команда не выиграла матч! https://example.org/news',),
    ('Лыжник не очень быстро прошёл дистанцию.',),
    ('[club123|Хоккейный клуб] выиграл со счётом 3:2.',),
    ('Хоккей хоккея хоккеем, футбол футбола футболом.',)
], ['text'])
demo_tokens = Pipeline(stages=text_stages()).fit(demo).transform(demo)
demo_stop = StopWordsRemover(inputCol='tokens', outputCol='default_stop', stopWords=RU_STOP).transform(demo_tokens)
demo_stop = StopWordsRemover(inputCol='tokens', outputCol='safe_stop', stopWords=SAFE_STOP).transform(demo_stop)
demo_stop.select('text', 'tokens', 'default_stop', 'safe_stop').show(truncate=False)
conclusion('Стандартный стоп-лист может удалить отрицание; SAFE_STOP сохраняет такие частицы. '
    'Пунктуация, числа и словоформы обрабатываются по фиксированным правилам. '
    'Влияние стоп-слов на качество будет измерено на valid до открытия test.')


## 5. Частоты, длины текстов и визуализация

Описательные частотные исследования выполняем только на исходных объектах train, до нарезки и ограничения числа фрагментов. TF - число вхождений токена, DF - число документов, в которых он встретился хотя бы раз. Относительная DF внутри категории позволяет сравнивать классы разного размера. До `toPandas()` данные агрегируются или ограничиваются сверху.


In [ ]:
eda = split_docs.filter("split = 'train'").select('row_id', 'category', 'label', 'text', 'tokens', 'n_tokens').cache()
length_stats = eda.groupBy('category').agg(F.count('*').alias('n'),
    F.avg('n_tokens').alias('mean_tokens'),
    F.expr('percentile_approx(n_tokens, array(0.5, 0.9, 0.99))').alias('quantiles'),
    F.max('n_tokens').alias('max_tokens')).orderBy('category').toPandas()
display(length_stats)
length_stats.to_csv(OUT / 'train_lengths.csv', index=False)
length_hist = eda.select(F.least((F.col('n_tokens') / 20).cast('int'), F.lit(50)).alias('bin')).groupBy('bin').count().orderBy('bin').toPandas()
fig, ax = plt.subplots()
ax.bar(length_hist['bin'] * 20, length_hist['count'], width=18)
ax.set(xlabel='Токены, интервалы по 20; последняя колонка ≥1000', ylabel='Число текстов', title='Train: распределение длины исходных текстов')
savefig('02_lengths.png')
occurrences = eda.select('row_id', 'category', F.explode('tokens').alias('term')).cache()
term_stats = occurrences.groupBy('term').agg(F.count('*').alias('tf'), F.countDistinct('row_id').alias('df')).cache()
top = (term_stats.filter(~F.col('term').isin(sorted(DISPLAY_STOP)))
    .orderBy(F.desc('df'), 'term').limit(20).toPandas())
display(top)
top.to_csv(OUT / 'train_top_terms.csv', index=False)
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top.term[::-1], top.df[::-1])
ax.set(xlabel='DF: число исходных train-текстов с токеном', title='Train: 20 частых токенов; стоп-слова скрыты только на рисунке')
savefig('03_terms_df.png')
ranked = term_stats.orderBy(F.desc('tf'), 'term').limit(10000).toPandas()
fig, ax = plt.subplots()
ax.loglog(np.arange(1, len(ranked) + 1), ranked.tf)
ax.set(xlabel='Ранг токена', ylabel='TF: число вхождений', title='Train: ранг–частота, до 10 000 токенов')
savefig('04_rank_frequency.png')
conclusion('Диаграммы показывают длины документов и концентрацию частот. '
    f'Среди показанных токенов наибольшая DF у **{top.iloc[0].term}** '
    f'(**{int(top.iloc[0].df):,}** текстов). Различие длин мотивирует сравнение BoW с нормированным TF–IDF.')


In [ ]:
selected_terms = ['матч', 'гол', 'команда', 'мяч', 'шайба', 'гонка', 'лыжи', 'теннис', 'турнир', 'не']
class_n = eda.groupBy('category').count().withColumnRenamed('count', 'class_n')
profile_small = (occurrences.filter(F.col('term').isin(selected_terms))
    .groupBy('term', 'category').agg(F.countDistinct('row_id').alias('df'))
    .join(class_n, 'category').withColumn('share', F.col('df') / F.col('class_n'))
    .select('term', 'category', 'share').toPandas())
profile = (profile_small.pivot(index='term', columns='category', values='share')
    .reindex(index=selected_terms, columns=CATEGORIES).fillna(0))
fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(profile.to_numpy(), aspect='auto', cmap='Blues')
ax.set_xticks(range(K), CATEGORIES, rotation=45, ha='right')
ax.set_yticks(range(len(profile)), profile.index)
ax.set(xlabel='Тематика', ylabel='Токен', title='Train: доля исходных текстов класса, содержащих токен')
fig.colorbar(im, ax=ax, label='Относительная DF')
savefig('05_class_terms.png')
profile.to_csv(OUT / 'train_class_term_profile.csv')
try:
    from wordcloud import WordCloud
    from matplotlib.font_manager import findfont
    cloud_rows = (term_stats.filter(F.col('term').rlike(r'^[а-яёa-z]+$'))
        .filter(~F.col('term').isin(sorted(DISPLAY_STOP)))
        .orderBy(F.desc('df'), 'term').limit(100).collect())
    cloud = WordCloud(font_path=findfont('DejaVu Sans'), width=1200, height=500,
        random_state=SEED, background_color='white').generate_from_frequencies({r.term: r.df for r in cloud_rows})
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(cloud, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Train: облако 100 токенов по DF')
    savefig('06_wordcloud.png')
except ImportError:
    print('Облако слов пропущено: установите дополнительный пакет wordcloud.')
conclusion('Относительная DF сравнивает темы независимо от числа текстов. '
    'Общеспортивные слова могут встречаться в нескольких классах, '
    'поэтому один токен не всегда позволяет однозначно определить тематику. '
    'Облако - иллюстрация; точные значения представлены в таблицах.')


## 6. Bag of Words, TF–IDF и разреженные векторы

Словарь задаёт координаты вектора; BoW содержит счётчики. В разреженном векторе хранятся размерность, индексы ненулевых координат и их значения. Плотная матрица `N × V` типа float64 занимает примерно `8NV` байт без учёта служебных объектов.

В Spark используется сглаженная формула:

$$\operatorname{IDF}(t)=\ln\frac{N_{train}+1}{DF_{train}(t)+1},\qquad z_j(d)=TF(t_j,d)\,IDF(t_j).$$

После TF–IDF L2-нормирование делит ненулевой вектор на его длину. Ни словарь, ни IDF не переобучаются на valid/test. На трёх маленьких документах ниже расчёт проверяется вручную; только эти векторы переводятся в плотный вид.


In [ ]:
toy = spark.createDataFrame([(0, 'команда гол гол'.split()),
    (1, 'команда не гол'.split()), (2, 'лыжи снег'.split())], ['id', 'tokens'])
TOY_VOCAB = ['команда', 'гол', 'не', 'лыжи', 'снег']
toy_cv = CountVectorizerModel.from_vocabulary(TOY_VOCAB, inputCol='tokens', outputCol='counts')
toy_counts = toy_cv.transform(toy)
expected = np.array([[1, 2, 0, 0, 0], [1, 1, 1, 0, 0], [0, 0, 0, 1, 1]])
actual = np.array([r.counts.toArray() for r in toy_counts.orderBy('id').collect()])
np.testing.assert_array_equal(actual, expected)
toy_idf = IDF(inputCol='counts', outputCol='tfidf').fit(toy_counts)
manual_idf = np.log(4 / (np.count_nonzero(expected, axis=0) + 1))
np.testing.assert_allclose(toy_idf.idf.toArray(), manual_idf)
normalized = Normalizer(inputCol='tfidf', outputCol='features', p=2.0).transform(toy_idf.transform(toy_counts))
man = expected * manual_idf
man = man / np.linalg.norm(man, axis=1, keepdims=True)
np.testing.assert_allclose(np.array([r.features.toArray() for r in normalized.orderBy('id').collect()]), man)
toy_counts.orderBy('id').show(truncate=False)
display(pd.DataFrame({'term': TOY_VOCAB, 'df': np.count_nonzero(expected, axis=0), 'idf': manual_idf}))
normalized.orderBy('id').show(truncate=False)
fig, ax = plt.subplots(figsize=(8, 3))
ax.imshow(actual, aspect='auto', cmap='Blues')
ax.set_xticks(range(5), TOY_VOCAB)
ax.set_yticks(range(3), ['d0', 'd1', 'd2'])
for i in range(3):
    for j in range(5):
        ax.text(j, i, int(actual[i,j]), ha='center', va='center')
ax.set(xlabel='Словарь', ylabel='Документ', title='Учебный пример BoW; не основной корпус')
savefig('07_bow_manual.png')
N_TRAIN = train.count()
print(f'Плотная матрица train при V={VOCAB_SIZE}: {8 * N_TRAIN * VOCAB_SIZE / 2**30:.2f} ГиБ')
conclusion('Ручные BoW, IDF и нормированные TF–IDF совпали с Spark. '
    'Слова «команда» и «гол» имеют IDF=ln(4/3), остальные - ln(2). '
    'Основная матрица признаков далее остаётся разреженной.')


In [ ]:
NGram(n=2, inputCol='tokens', outputCol='bigrams').transform(toy).select('tokens', 'bigrams').show(truncate=False)
HashingTF(inputCol='tokens', outputCol='hashed', numFeatures=2**16).transform(toy).select('id', 'hashed').show(truncate=False)
conclusion('Биграммы добавляют признаки соседства, например «не гол». '
    'HashingTF не обучает словарь и учитывает новые токены через хеш, но допускает коллизии. '
    'В основных опытах выбран CountVectorizer, позволяющий интерпретировать координаты через слова.')


## 7. Модели и метрики

| Модель | Признаки | Классификатор |
|---|---|---|
| NB_BOW | Униграммы, счётчики | Мультиномиальный наивный Байес, smoothing=1 |
| LR_BOW | Униграммы, счётчики | Мультиномиальная логистическая регрессия |
| LR_TFIDF | Униграммы, TF–IDF + L2 | Мультиномиальная логистическая регрессия |
| LR_TFIDF_BIGRAM | Униграммы и биграммы, TF–IDF + L2 | Мультиномиальная логистическая регрессия |

Все варианты - полные Pipeline: от столбца `text` до вероятностей 13 классов. Для LR используется L2-регуляризация `regParam=0.001`, `standardization=False`. `minDF=3` отсекает признаки, встречающиеся менее чем в трёх обучающих фрагментах; `vocabSize=20000` ограничивает словарь. Биграммы строятся до удаления стоп-слов, чтобы не создавать искусственное соседство.

Основной критерий выбора - **macro-F1 на valid по группам oid**: задача относится к тематике источника публикаций. Вероятности сначала усредняются по фрагментам поста, затем поровну по постам одного oid. Рядом приводим macro-F1 по фрагментам. Постоянное правило всегда предсказывает самый частый класс train-групп oid.

Матрица ошибок строится в Spark, а в Python попадают максимум 13×13 счётчиков. Для класса `k`: `precision=TP/predicted`, `recall=TP/support`, `F1=2PR/(P+R)`. Macro-F1 усредняет 13 F1 поровну; weighted-F1 учитывает размер классов. Нулевые знаменатели дают 0. Log loss оценивает вероятности правильного класса; меньше - лучше. Метрика `f1` встроенного MulticlassClassificationEvaluator означает weighted-F1, поэтому macro-F1 считаем явно.


In [ ]:
SPECS = ['NB_BOW', 'LR_BOW', 'LR_TFIDF', 'LR_TFIDF_BIGRAM']

def make_pipeline(name, stop_words=None):
    if name not in SPECS:
        raise ValueError(name)
    stages = text_stages()
    input_col = 'tokens'
    if stop_words is not None:
        if name.endswith('BIGRAM'):
            raise ValueError('В этом опыте удаление стоп-слов сравнивается только для униграмм')
        stages.append(StopWordsRemover(inputCol='tokens', outputCol='filtered', stopWords=stop_words))
        input_col = 'filtered'
    if name.endswith('BIGRAM'):
        stages += [NGram(n=2, inputCol='tokens', outputCol='bigrams'),
            SQLTransformer(statement='SELECT *, concat(tokens, bigrams) AS terms FROM __THIS__')]
        input_col = 'terms'
    tfidf = 'TFIDF' in name
    stages.append(CountVectorizer(inputCol=input_col, outputCol='counts' if tfidf else 'features',
        vocabSize=VOCAB_SIZE, minDF=MIN_DF, binary=False))
    if tfidf:
        stages += [IDF(inputCol='counts', outputCol='tfidf'),
            Normalizer(inputCol='tfidf', outputCol='features', p=2.0)]
    classifier = (NaiveBayes(modelType='multinomial', smoothing=1.0) if name == 'NB_BOW' else
        LogisticRegression(family='multinomial', maxIter=150, tol=1e-6,
            regParam=REG_PARAM, elasticNetParam=0.0, standardization=False))
    return Pipeline(stages=stages + [classifier])

def metrics(pred):
    cm = np.zeros((K, K), dtype=np.int64)
    for r in pred.groupBy('label', 'prediction').count().collect():
        cm[int(r.label), int(r.prediction)] = r['count']
    assert cm.sum() > 0, 'Пустая выборка'
    support, predicted = cm.sum(axis=1), cm.sum(axis=0)
    tp = np.diag(cm).astype(float)
    precision = np.divide(tp, predicted, out=np.zeros(K), where=predicted > 0)
    recall = np.divide(tp, support, out=np.zeros(K), where=support > 0)
    f1 = np.divide(2*precision*recall, precision+recall, out=np.zeros(K), where=(precision+recall)>0)
    result = dict(accuracy=float(tp.sum()/cm.sum()), macro_f1=float(f1.mean()),
        weighted_f1=float(f1 @ support/cm.sum()), n=int(cm.sum()))
    if 'probability' in pred.columns:
        s = pred.withColumn('_p', vector_to_array('probability'))
        true_p = F.expr('element_at(_p, cast(label as int) + 1)')
        result['log_loss'] = float(s.agg(F.avg(-F.log(F.greatest(true_p, F.lit(1e-15))))).first()[0])
    per_class = pd.DataFrame(dict(category=CATEGORIES, support=support, precision=precision, recall=recall, f1=f1))
    return result, per_class, cm

def aggregate_objects(pred):
    # Один прогноз исходному текстовому объекту: средняя вероятность его фрагментов.
    p = pred.withColumn('_prob', vector_to_array('probability'))
    agg = p.groupBy('row_id', 'label', 'category').agg(
        *[F.avg(F.col('_prob')[i]).alias(f'p{i}') for i in range(K)])
    agg = agg.withColumn('_p', F.array(*[F.col(f'p{i}') for i in range(K)]))
    agg = agg.withColumn('prediction', (F.expr('array_position(_p, array_max(_p))')-1).cast('double'))
    from pyspark.ml.functions import array_to_vector
    return agg.withColumn('probability', array_to_vector('_p')).drop('_p')

majority = float(split_docs.filter("split = 'train'").select('source_id','label').distinct().groupBy('label').count()
    .orderBy(F.desc('count'), 'label').first().label)
valid_objects = split_docs.filter("split = 'valid'").select('source_id', 'category', 'label').distinct()
baseline_valid, _, _ = metrics(valid_objects.withColumn('prediction', F.lit(majority)))
print('Постоянный класс:', LABEL_NAMES[int(majority)], '| valid:', baseline_valid)


def aggregate_sources(object_pred):
    # Сначала каждый исходный пост получает один голос, затем усреднение по oid.
    post = (object_pred.join(split_docs.select('row_id','source_id'),'row_id')
        .withColumn('_prob',vector_to_array('probability')))
    grouped = post.groupBy('source_id','label','category').agg(*[
        F.avg(F.col('_prob')[i]).alias(f'p{i}') for i in range(K)])
    grouped = grouped.withColumn('_p',F.array(*[F.col(f'p{i}') for i in range(K)]))
    grouped = grouped.withColumn('prediction',(F.expr('array_position(_p,array_max(_p))')-1).cast('double'))
    from pyspark.ml.functions import array_to_vector
    return grouped.withColumn('probability',array_to_vector('_p')).drop('_p')


## 8. Обучение и сравнение на valid

Каждый Pipeline обучается только на train. Помимо четырёх основных решений сравниваются два стоп-листа для LR_TFIDF. Все шесть вариантов входят в общий выбор **до** открытия test. Время измеряется для полного `fit`, а не только классификатора. Последовательное выполнение ограничивает одновременное потребление памяти; в кэше не сохраняются предсказания всех моделей.


In [ ]:
models, rows, validation_classes = {}, [], []
experiments = [(name, name, None) for name in SPECS]
if RUN_STOP_ABLATION:
    experiments += [('LR_TFIDF_DEFAULT_STOP', 'LR_TFIDF', RU_STOP),
                    ('LR_TFIDF_SAFE_STOP', 'LR_TFIDF', SAFE_STOP)]
for display_name, spec, stop in experiments:
    sc.setJobGroup('PR6_' + display_name, 'Обучение ' + display_name)
    started = perf_counter()
    model = make_pipeline(spec, stop_words=stop).fit(train)
    fit_s = perf_counter() - started
    pred = model.transform(valid).cache()
    pred.count()
    fragment_metrics, _, _ = metrics(pred)
    obj = aggregate_objects(pred).cache()
    object_metrics, _, _ = metrics(obj)
    source_metrics, classes, _ = metrics(aggregate_sources(obj))
    classes['model'] = display_name
    validation_classes.append(classes)
    cv = next(s for s in model.stages if isinstance(s, CountVectorizerModel))
    rows.append(dict(model=display_name, vocabulary=len(cv.vocabulary), fit_s=fit_s,
        fragment_macro_f1=fragment_metrics['macro_f1'], object_macro_f1=object_metrics['macro_f1'], **source_metrics))
    models[display_name] = model
    print(display_name, '| vocab:', len(cv.vocabulary), '| object macro-F1:', round(source_metrics['macro_f1'],4), '| fit:', round(fit_s,1), 'с')
    obj.unpersist()
    pred.unpersist()
sc.setLocalProperty('spark.jobGroup.id', None)
comparison = pd.DataFrame(rows).sort_values(['macro_f1','model'], ascending=[False,True]).reset_index(drop=True)
display(comparison.round(4))
comparison.to_csv(OUT / 'validation.csv', index=False)
pd.concat(validation_classes).to_csv(OUT / 'validation_by_class.csv', index=False)
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(comparison.model, comparison.macro_f1)
ax.axhline(baseline_valid['macro_f1'], linestyle='--', color='red', label='Постоянный класс')
ax.set(ylabel='Macro-F1 по группам oid', ylim=(0,1), title='Valid: сравнение полных Pipeline')
ax.tick_params(axis='x', rotation=25)
ax.legend()
savefig('08_validation.png')
BEST_NAME = str(comparison.iloc[0]['model'])
BEST_MODEL = models[BEST_NAME]
conclusion(f'По valid выбран **{BEST_NAME}**: macro-F1 по группам oid '
    f'**{comparison.iloc[0].macro_f1:.4f}**, у постоянного правила '
    f'**{baseline_valid["macro_f1"]:.4f}**. '
    'Остальные параметры и способ обработки после этой точки фиксируются.')
if RUN_STOP_ABLATION:
    stop_comparison = comparison[comparison.model.isin(['LR_TFIDF','LR_TFIDF_DEFAULT_STOP','LR_TFIDF_SAFE_STOP'])]
    display(stop_comparison[['model','vocabulary','macro_f1','fit_s']].round(4))
    winner_stop = stop_comparison.sort_values(['macro_f1','model'],ascending=[False,True]).iloc[0]
    conclusion(f'Среди вариантов стоп-слов лучший на valid - **{winner_stop.model}**. '
        'Вывод относится к этому разбиению и заданной регуляризации; '
        'он не доказывает универсального преимущества одного стоп-листа.')


## 9. Итоговый тест, покрытие словаря и ошибки

Test открывается после завершения сравнения. Проверяем только выбранную модель и постоянное правило. Выводим три уровня: фрагмент, пост и группа oid. Сначала измеряем качество по фрагментам, затем усредняем вероятности по исходному текстовому объекту. Повторного обучения на train+valid здесь нет.

Для модели с биграммами OOV считается по объединению униграмм и биграмм. `nnz` - фактическое число ненулевых координат конечного вектора: IDF способен занулить координату известного токена. Пустые или полностью неизвестные тексты диагностируются отдельно.


In [ ]:
test_predictions = BEST_MODEL.transform(test).cache()
test_predictions.count()
TEST_FRAG_METRICS, TEST_FRAG_CLASSES, TEST_FRAG_CM = metrics(test_predictions)
test_object_predictions = aggregate_objects(test_predictions).cache()
TEST_OBJECT_METRICS, TEST_OBJECT_CLASSES, TEST_OBJECT_CM = metrics(test_object_predictions)
test_source_predictions = aggregate_sources(test_object_predictions).cache()
TEST_SOURCE_METRICS, TEST_SOURCE_CLASSES, TEST_SOURCE_CM = metrics(test_source_predictions)
baseline_test_source, _, _ = metrics(split_docs.filter("split = 'test'").select('source_id','category','label').distinct().withColumn('prediction',F.lit(majority)))
baseline_test_frag, _, _ = metrics(test.withColumn('prediction', F.lit(majority)))
baseline_test_obj, _, _ = metrics(split_docs.filter("split = 'test'").withColumn('prediction', F.lit(majority)))
test_summary = pd.DataFrame([
    {'model': BEST_NAME, 'level':'source_oid', **TEST_SOURCE_METRICS},
    {'model': 'majority', 'level':'source_oid', **baseline_test_source},
    {'model': BEST_NAME, 'level':'fragment', **TEST_FRAG_METRICS},
    {'model': BEST_NAME, 'level':'object', **TEST_OBJECT_METRICS},
    {'model':'majority', 'level':'fragment', **baseline_test_frag},
    {'model':'majority', 'level':'object', **baseline_test_obj}])
display(test_summary.round(4))
display(TEST_OBJECT_CLASSES.round(4))
display(TEST_SOURCE_CLASSES.round(4))
TEST_SOURCE_CLASSES.to_csv(OUT / "test_source_by_class.csv",index=False)
test_summary.to_csv(OUT / 'test_summary.csv', index=False)
TEST_OBJECT_CLASSES.to_csv(OUT / 'test_object_by_class.csv', index=False)
TEST_FRAG_CLASSES.to_csv(OUT / 'test_fragment_by_class.csv', index=False)
normalized_cm = TEST_SOURCE_CM / np.maximum(TEST_SOURCE_CM.sum(axis=1,keepdims=True),1)
fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(normalized_cm, vmin=0, vmax=1, cmap='Blues')
for i in range(K):
    for j in range(K):
        if TEST_SOURCE_CM[i,j]:
            ax.text(j,i,f'{TEST_SOURCE_CM[i,j]}\n{normalized_cm[i,j]:.0%}', ha='center',va='center',fontsize=7,
                color='white' if normalized_cm[i,j] > .6 else 'black')
ax.set_xticks(range(K), CATEGORIES, rotation=60,ha='right')
ax.set_yticks(range(K), CATEGORIES)
ax.set(xlabel='Предсказанная тематика',ylabel='Истинная тематика',title=f'Test, группы oid: {BEST_NAME}')
fig.colorbar(im,ax=ax,label='Доля внутри истинного класса')
savefig('09_confusion_matrix.png')
worst = TEST_OBJECT_CLASSES.sort_values(['f1','category']).iloc[0]
conclusion(f'На test macro-F1 по фрагментам - **{TEST_FRAG_METRICS["macro_f1"]:.4f}**, '
    f'по исходным объектам - **{TEST_OBJECT_METRICS["macro_f1"]:.4f}**. '
    f'Наименьший F1 у **{worst.category}** (**{worst.f1:.4f}**). '
    'Различия уровней связаны с усреднением вероятностей и вкладом длинных публикаций.')

conclusion(f'Macro-F1 по тестовым группам oid: **{TEST_SOURCE_METRICS["macro_f1"]:.4f}**; постоянное правило: **{baseline_test_source["macro_f1"]:.4f}**. Усреднение по постам даёт каждой группе источника один ответ.')


In [ ]:
cv_best = next(s for s in BEST_MODEL.stages if isinstance(s, CountVectorizerModel))
terms_col = cv_best.getInputCol()
vocab_df = spark.createDataFrame([(w,) for w in cv_best.vocabulary], ['term'])
all_terms = test_predictions.select(F.explode(terms_col).alias('term'))
TERM_N = all_terms.count()
OOV_N = all_terms.join(F.broadcast(vocab_df),'term','left_anti').count()
OOV_SHARE = OOV_N / max(TERM_N, 1)
@F.udf(T.IntegerType())
def nnz(v):
    return int(v.numNonzeros())
coverage = test_predictions.withColumn('nnz', nnz('features'))
ZERO_VECTORS = coverage.filter('nnz = 0').count()
MEAN_NNZ = float(coverage.agg(F.avg('nnz')).first()[0])
coverage.filter('nnz = 0').select('text','label','prediction').show(5, truncate=150)
conclusion(f'OOV среди вхождений тестовых признаковых единиц - **{OOV_SHARE:.2%}**; '
    f'нулевых фрагментов - **{ZERO_VECTORS:,}**, среднее nnz - **{MEAN_NNZ:.1f}**. '
    'Для нулевого вектора решение определяется свободными членами LR или априорными вероятностями NB.')


### Разбор ошибок на уровне исходных объектов

Покажем до 12 уверенных ошибочных предсказаний целых текстов. Для каждой записи выводятся исходная и предсказанная темы, обе вероятности, текст, диагностические лексические признаки. Анализ охватывает не менее пяти ошибок, если они есть. При меньшем числе разбираются все имеющиеся. Лексическое пересечение - гипотеза причины ошибки; подтверждение требует прочтения текста и проверки разметки. Метки test не исправляются по ответам модели.


In [ ]:
error_objects = (test_object_predictions.filter(F.col('label') != F.col('prediction'))
    .withColumn('_p', vector_to_array('probability'))
    .withColumn('confidence', F.array_max('_p'))
    .withColumn('true_probability', F.expr('element_at(_p, cast(label as int) + 1)'))
    .join(split_docs.select('row_id','text'), 'row_id')
    .orderBy(F.desc('confidence'),'row_id')
    .select('row_id','text','category','prediction','confidence','true_probability').limit(12).toPandas())
LEXICON = {
    'football': {'гол','футбол','ворота','матч'}, 'hockey': {'шайба','хоккей','лёд','лед','матч'},
    'basketball': {'баскетбол','кольцо','мяч','матч'}, 'volleyball': {'волейбол','сетка','мяч','матч'},
    'tennis': {'теннис','сет','корт','матч'}, 'winter_sport': {'лыжи','снег','биатлон','коньки'},
    'autosport': {'гонка','болид','трасса'}, 'motosport': {'гонка','мотоцикл','трасса'},
    'athletics': {'бег','марафон','прыжок'}, 'boardgames': {'шахматы','доска','партия'},
    'esport': {'dota','cs','киберспорт','турнир'}, 'extreme': {'скейт','трюк','прыжок'},
    'martial_arts': {'бой','бокс','раунд','удар'}}
if len(error_objects):
    error_objects['predicted_category'] = error_objects.prediction.astype(int).map(LABEL_NAMES)
    def diagnostic(row):
        terms = set(re.findall(r'[а-яёa-z0-9]+', row.text.lower()))
        true_hits = sorted(terms & LEXICON.get(row.category,set()))
        pred_hits = sorted(terms & LEXICON.get(row.predicted_category,set()))
        flags = []
        if len(terms) < 8:
            flags.append('мало лексического контекста')
        if true_hits and pred_hits:
            flags.append('есть лексика обеих тематик')
        if not true_hits:
            flags.append('нет точных совпадений с небольшим диагностическим словарём истинной темы')
        return f'Точная лексика метки: {true_hits}; прогноза: {pred_hits}. ' + '; '.join(flags)
    error_objects['diagnostic'] = error_objects.apply(diagnostic,axis=1)
    display(error_objects[['category','predicted_category','confidence','true_probability','diagnostic']])
    for i, row in enumerate(error_objects.itertuples(),1):
        # text печатается как текст, без интерпретации разметки исходной публикации.
        print(f'\nОшибка {i} | {row.category} → {row.predicted_category}')
        print(f'p(прогноз)={row.confidence:.4f}; p(метка)={row.true_probability:.4f}')
        print(row.text)
        print('Диагностика:',row.diagnostic)
    error_objects.to_csv(OUT / 'error_examples.csv',index=False)
    conclusion(f'Показано **{len(error_objects)}** ошибочных объектов с вероятностями и диагностикой. '
        'Общеспортивная лексика, короткий контекст, упоминания нескольких дисциплин '
        'и метка тематики паблика вместо темы конкретного поста могут объяснять часть ошибок. '
        'Автоматические признаки не устанавливают истинную причину.')
else:
    conclusion('В этом тестовом запуске ошибочных исходных объектов не обнаружено. '
        'Это не гарантирует безошибочную работу на будущих публикациях.')


### Содержательный разбор пяти ошибок выполненного запуска

Ниже приведены гипотезы по самим текстам. Они дополняют автоматическую диагностику; исходные тестовые метки сохранены. Номера и вероятности привязаны к сохранённому запуску, при изменении окружения или параметров примеры необходимо сверить заново.

| Объект (начало row_id) | Исходная тема → прогноз | Разбор |
|---|---|---|
| `025f7fa52a46` | motosport → autosport | В публикации о MotoGP есть сильный маркер мотоспорта, но большая часть текста описывает старт, гонку, круги, резину и времена. Такая лексика общая для двух классов. Ошибка показывает, что сумма общих признаков может перевесить специализированный маркер. Следующий опыт на train/valid - проверить покрытие названий серий и нормализацию имён; test после этого не использовать для выбора. |
| `03209e1aba7b` | martial_arts → basketball | Публикация рекламирует педагогическое образование. Она упоминает бокс и кикбоксинг, однако большая часть текста посвящена документам, поступлению, стоимости и обучению детей. Тема рекламы расходится с основной спортивной лексикой паблика. По одному такому посту определять специализацию источника трудно; для этого и рассчитывается отдельная оценка по нескольким постам oid. |
| `03cc2d1c8061` | motosport → autosport | Сюжет посвящён пилоту MotoGP и борьбе за подиум. Слова о Гран-при, чемпионстве, стартовой позиции и скорости применимы и к автогонкам. Упоминания фамилий и обезличенные формы названий команд могут хуже переноситься на новый источник. Это гипотеза по тексту: причинный вклад конкретных слов отдельно не доказан. |
| `0421343383e1` | motosport → autosport | Описана гонка Квартараро; в тексте есть Yamaha и упоминание байка. Ошибка autosport при наличии специфической информации показывает ограничение мешка слов и частотной модели: она не проверяет, что объектом гонки является мотоцикл. Усреднение других публикаций источника иногда исправляет отдельный ошибочный прогноз, что оценивается отдельно. |
| `04bfd4451226` | motosport → autosport | Текст об Эспаргаро прямо описывает неисправность и смену мотоцикла, но содержит много общих гоночных слов: круг, очки, чемпионат, двигатель, покрышка. Для различения видов спорта важно сочетание признаков, а не только общая тема гонок. Повтор такого типа ошибок указывает на устойчивую путаницу соседних классов, а не единичную случайность. |

Уверенные ошибки NB требуют особого внимания: предположение об условной независимости слов не выполняется буквально, а повторяющаяся и связанная лексика способна давать чрезмерно резкие вероятности. Поэтому высокий максимум probability не считается гарантией правильного ответа; рядом рассчитаны log loss и вероятность исходной метки.

В исходных текстах встречаются технические формы с `tokenoid`. Они сохраняются как часть выданного корпуса и могут становиться признаками. Их удаление или отдельная нормализация - возможная абляция на train/valid, но её влияние в данном опыте не измерялось.

## 10. Интерпретация коэффициентов

Интерпретируем фиксированную LR_TFIDF, даже если победила другая модель. Для пары «футбол / хоккей» разность коэффициентов `w_football − w_hockey` показывает направление вклада токена в логарифм отношения вероятностей при прочих фиксированных координатах. Это не причинное влияние и не универсальная важность слова. Коэффициенты относятся к нормированным TF–IDF-признакам.


In [ ]:
lr_pipe = models['LR_TFIDF']
lr_cv = next(s for s in lr_pipe.stages if isinstance(s, CountVectorizerModel))
weights = lr_pipe.stages[-1].coefficientMatrix.toArray()  # Только K × V коэффициентов, не корпус.
a, b = CATEGORIES.index('football'), CATEGORIES.index('hockey')
contrast = weights[a] - weights[b]
idx = np.unique(np.r_[np.argsort(contrast)[:10],np.argsort(contrast)[-10:]])
coef_table = pd.DataFrame({'term':np.array(lr_cv.vocabulary)[idx], 'contrast':contrast[idx]}).sort_values('contrast')
display(coef_table)
fig, ax = plt.subplots(figsize=(9,7))
ax.barh(coef_table.term,coef_table.contrast,color=['#be5a50' if v < 0 else '#377eb8' for v in coef_table.contrast])
ax.axvline(0,color='black',linewidth=.8)
ax.set(xlabel='w(football) − w(hockey)',title='LR_TFIDF, train: различающие коэффициенты двух классов')
savefig('10_coefficients.png')
coef_table.to_csv(OUT / 'coefficient_contrast.csv',index=False)
conclusion('Положительный контраст увеличивает оценку football относительно hockey, '
    'отрицательный - наоборот. Имена команд, спортсменов и названия турниров могут '
    'отражать конкретную выборку источников, поэтому переносимость таких признаков ограничена.')


## 11. Генерация текста: цепи Маркова первого и второго порядка

Для генерации выбираем **одну тематику** - football - и до 300 исходных train-текстов по устойчивому хешу. Это аналог генератора по произведениям одного автора из методички. Корпус ограничен заранее, чтобы таблицы переходов помещались на драйвере. Длинные тексты не соединяются друг с другом.

Текст делится на предложения по переводу строки либо пробелу после `.?!`. Словоформы, стоп-слова и основные знаки препинания сохраняются. Правило границ эвристическое: сокращения, десятичные дроби и инициалы могут обрабатываться неточно. Для переходов выбираются предложения от 3 до 80 токенов.

Для порядка `q` оценивается `P(w|h)=count(h,w)/Σᵤcount(h,u)`. В первом порядке контекст - один токен, во втором - два. Перед предложением ставится `q` маркеров `<BOS>`, в конце - `<EOS>`. Подсчёт через NGram и groupBy выполняется в Spark. Генерация короткой последовательности - в Python.


In [ ]:
GEN_CATEGORY = 'football'
MARKOV_PATTERN = r"[а-яёa-z0-9]+(?:[-’'][а-яёa-z0-9]+)*|[.,!?;:--]|[()]{2,}"
markov_source = (split_docs.filter((F.col('split')=='train') & (F.col('category')==GEN_CATEGORY))
    .orderBy(F.sha2('row_id',256),'row_id').limit(MARKOV_DOCS))
markov_source.select('row_id').write.mode('overwrite').parquet((OUT / 'markov_source_manifest').as_posix())
# Сохраняем словоформы и пунктуацию; удаляем адреса и VK-технические идентификаторы.
markov_clean = SQLTransformer(statement=CLEAN_SQL).transform(markov_source.select('row_id','text'))
sentences = markov_clean.select('row_id', F.posexplode(
    F.split('text_norm',r'(?<=[.!?])\s+|[\r\n]+')).alias('sentence_id','sentence'))
sentences = (RegexTokenizer(inputCol='sentence',outputCol='words',pattern=MARKOV_PATTERN,
    gaps=False,toLowercase=True,minTokenLength=1).transform(sentences)
    .filter((F.size('words')>=3)&(F.size('words')<=80)).cache())
N_SENTENCES = sentences.count()
assert N_SENTENCES > 0, 'Нет предложений для генератора'
sentences.select('sentence','words').show(3,truncate=130)

def count_transitions(sentence_df, order):
    if order not in (1,2):
        raise ValueError(order)
    padded = sentence_df.withColumn('seq',F.concat(
        F.array(*[F.lit('<BOS>') for _ in range(order)]),F.col('words'),F.array(F.lit('<EOS>'))))
    grams = NGram(n=order+1,inputCol='seq',outputCol='grams').transform(padded)
    parts = F.split('gram',' ')
    return (grams.select(F.explode('grams').alias('gram')).select(F.lit(order).alias('order'),
        F.concat_ws(' ',F.slice(parts,1,order)).alias('context'),
        F.element_at(parts,order+1).alias('next'))
        .groupBy('order','context','next').count())

transitions = count_transitions(sentences,1).unionByName(count_transitions(sentences,2)).cache()
structure = transitions.groupBy('order').agg(F.countDistinct('context').alias('states'),F.count('*').alias('edges')).orderBy('order').toPandas()
display(structure)
transitions.write.mode('overwrite').parquet((OUT / 'markov_edges').as_posix())
if transitions.limit(MAX_EDGES+1).count()>MAX_EDGES:
    raise MemoryError('Таблица превышает MAX_EDGES. Уменьшите MARKOV_DOCS и повторите раздел 11.')
edge_rows = [r.asDict() for r in transitions.collect()]
(OUT / 'markov_edges.json').write_text(json.dumps(edge_rows,ensure_ascii=False),encoding='utf-8')
conclusion(f'Генератор обучен на **{N_SENTENCES:,}** train-предложениях тематики **{GEN_CATEGORY}**. '
    'Таблица показывает число состояний и переходов двух порядков. '
    'Переходы не пересекают границы предложений или исходных объектов.')


### Выбор продолжения, backoff и температура

Продолжения сортируются по токену, чтобы порядок результатов Spark не влиял на случайный выбор. Используется отдельный `random.Random(seed)`. Температура `T>0` задаёт веса `exp(log(count)/T)`: меньшая температура усиливает частые переходы.

Если контекст второго порядка неизвестен, ищется переход по последнему слову. Если неизвестно и оно, используется стартовое состояние. Возвращаются длина продолжения, число backoff-обращений и причина остановки. `max_tokens` ограничивает число **новых** токенов, префикс считается отдельно. Стартовый переход при неизвестном префиксе обеспечивает завершение, но не гарантирует смысловую связь с префиксом.


In [ ]:
def build_chain(records):
    chain = defaultdict(list)
    for r in records:
        chain[(int(r['order']), tuple(r['context'].split(' ')))].append((r['next'],int(r['count'])))
    return {key:sorted(options) for key,options in chain.items()}

def generate(chain,prompt='',order=2,max_tokens=40,seed=42,temperature=1.0):
    if order not in (1,2) or max_tokens<1 or not math.isfinite(temperature) or temperature<=0:
        raise ValueError('Некорректные параметры генерации')
    rng = random.Random(seed)
    prefix = re.findall(MARKOV_PATTERN,prompt.lower())
    history = ['<BOS>']*order+prefix
    result, backoffs = [], 0
    reason = 'max_tokens'
    for _ in range(max_tokens):
        options = chain.get((order,tuple(history[-order:])))
        if not options and order==2:
            options = chain.get((1,(history[-1],)))
            backoffs += 1
        if not options:
            options = chain.get((order,('<BOS>',)*order))
            backoffs += 1
        if not options:
            raise ValueError('Нет стартового состояния')
        words, counts = zip(*options)
        logs = [math.log(c)/temperature for c in counts]
        weights = [math.exp(v-max(logs)) for v in logs]
        word = rng.choices(words,weights=weights,k=1)[0]
        if word=='<EOS>':
            reason='eos'
            break
        result.append(word)
        history.append(word)
    text = re.sub(r'\s+([.,!?;:])',r'\1',' '.join(prefix+result))
    return dict(text=text,prefix_tokens=len(prefix),generated_tokens=len(result),
                backoffs=backoffs,stopped_by=reason)

chain = build_chain(edge_rows)
generations = []
for order in (1,2):
    for gen_seed in (7,42,101):
        result = generate(chain,order=order,seed=gen_seed)
        generations.append(dict(order=order,seed=gen_seed,**result))
generation_table = pd.DataFrame(generations)
display(generation_table)
for r in generations:
    print(f'Порядок {r["order"]}, seed={r["seed"]}: {r["text"]}')
generation_table.to_csv(OUT / 'markov_generations.csv',index=False)

# Доля точных совпадений с обучающими предложениями - проверка воспроизведения.
training_sentences = sentences.select(F.concat_ws(' ','words').alias('canonical')).distinct()
generated_df = spark.createDataFrame([(r['order'],r['seed'], ' '.join(re.findall(MARKOV_PATTERN,r['text'])))
    for r in generations],['order','seed','canonical'])
matched = generated_df.join(training_sentences,'canonical','left_semi').count()
conclusion(f'Получено шесть текстов для двух порядков и seed=7,42,101. '
    f'Полных точных совпадений с выбранными train-предложениями: **{matched}/6**. '
    'Второй порядок учитывает более длинный контекст, но способен воспроизводить исходные фразы. '
    'Оба генератора моделируют локальное соседство и не проверяют факты или связность сюжета.')


### Разбор фактической генерации

В первом порядке средняя длина трёх примеров - **4.7 токена**, во втором - **22.3**. Все шесть примеров завершились по EOS и не потребовали backoff. Отдельные проверки ниже принудительно проверяют неизвестный контекст и ограничение длины.

- **Первый порядок, seed=7:** получилась предельно короткая фраза с названием клуба и точкой. Локальные переходы быстро привели к концу предложения.
- **Первый порядок, seed=42:** в одной фразе соединены субъект публикации и выражение времени. Грамматически и семантически связь нарушена: одного предшествующего слова недостаточно для выбора осмысленного продолжения.
- **Первый порядок, seed=101:** получился короткий заголовок о претендентах и чемпионате. Он сохраняет тематическую лексику, но не определяет, о какой именно номинации идёт речь.
- **Второй порядок, seed=7:** последовательность стала длиннее, однако название трансфера сменилось высказыванием от первого лица. Локальная связность не обеспечивает единства говорящего и сюжета.
- **Второй порядок, seed=42:** соединены несколько трансферных новостей, поменялись клубы и участники. Длинные знакомые сочетания выглядят естественнее на малом участке, но цельный смысл потерян.
- **Второй порядок, seed=101:** получился связный заголовок о выборе лучшего игрока. По одному удачному примеру нельзя оценивать генератор в целом; для него отдельно проверено точное совпадение с обучающими предложениями.

Второй порядок в этом запуске чаще даёт развёрнутые фразы, но глобальная связность остаётся ограниченной. Увеличение длины не является самостоятельной метрикой качества языка.


In [ ]:
# Вероятности суммируются в 1 внутри каждого состояния.
prob_edges = transitions.withColumn('probability',F.col('count')/F.sum('count').over(Window.partitionBy('order','context')))
max_sum_error = (prob_edges.groupBy('order','context').agg(F.sum('probability').alias('s'))
    .agg(F.max(F.abs(F.col('s')-1))).first()[0])
assert max_sum_error < 1e-10
assert transitions.filter(F.col('context').contains('<EOS>') | (F.col('next')=='<BOS>')).count()==0
for order in (1,2):
    assert generate(chain,order=order,seed=42)==generate(chain,order=order,seed=42)
    assert generate(chain,order=order,max_tokens=10)['generated_tokens']<=10

# Проверка поведения генератора на известных и неизвестных контекстах.
known = next(context for (order,context) in sorted(chain) if order==2 and '<BOS>' not in context)
print('Известный контекст:',known,generate(chain,prompt=' '.join(known),order=2))
unknown = generate(chain,prompt='несуществующийконтекст абракадабратокен',order=2)
assert unknown['backoffs']>=1
print('Неизвестный контекст:',unknown)

# Детерминированные небольшие цепи проверяют обе причины остановки без надежды на случайность.
early_stop = {(1,('<BOS>',)):[('гол',1)],(1,('гол',)):[('<EOS>',1)]}
endless = {(1,('<BOS>',)):[('гол',1)],(1,('гол',)):[('гол',1)]}
assert generate(early_stop,order=1,max_tokens=10)['stopped_by']=='eos'
assert generate(endless,order=1,max_tokens=5)['stopped_by']=='max_tokens'
assert generate(endless,order=1,max_tokens=5)['generated_tokens']==5
# Граница предложений: недопустим переход из конца первой фразы в начало второй.
boundary_demo = spark.createDataFrame([(['гол','забил'],),(['матч','окончен'],)],['words'])
check_edges = count_transitions(boundary_demo,1)
assert check_edges.filter((F.col('context')=='забил') & (F.col('next')=='матч')).count()==0

# График по самому частому содержательному состоянию первого порядка.
contexts = transitions.filter((F.col('order')==1)&(~F.col('context').isin(['<BOS>']+sorted(DISPLAY_STOP))))
context = contexts.groupBy('context').agg(F.sum('count').alias('n')).orderBy(F.desc('n'),'context').first().context
options = chain[(1,(context,))]
total = sum(c for _,c in options)
top_next = sorted(options,key=lambda x:(-x[1],x[0]))[:12]
fig, ax = plt.subplots(figsize=(9,5))
ax.barh([w for w,c in top_next][::-1],[c/total for w,c in top_next][::-1])
ax.set(xlabel=f'P(следующий токен | {context})',title=f'Train/{GEN_CATEGORY}: цепь первого порядка, 12 продолжений')
savefig('11_markov_probabilities.png')
prob_edges.write.mode('overwrite').parquet((OUT / 'markov_probabilities').as_posix())
conclusion('Проверены суммы вероятностей, границы, воспроизводимость, неизвестный контекст, '
    'backoff и обе причины остановки: EOS и ограничение длины. '
    'На рисунке показаны вероятности части продолжений; их сумма может быть меньше единицы.')


## 12. Сохранение решения и применение

Сохраняется полный PipelineModel выбранного классификатора: очистка, токенизация, необязательный стоп-лист, словарь, IDF/нормирование при их наличии и модель. Рядом записываются настройки, соответствие числовых меток темам, хеш исходного CSV, разбиение, таблицы, графики и переходы генератора.

При обработке новых длинных публикаций используется та же нарезка и усреднение вероятностей. Для пустого текста флаг `has_features=False` явно показывает, что прогноз получен без распознанных признаков. Такие ответы нельзя трактовать как содержательную классификацию.


In [ ]:
MODEL_PATH = OUT / 'best_pipeline'
BEST_MODEL.write().overwrite().save(MODEL_PATH.as_posix())
loaded = PipelineModel.load(MODEL_PATH.as_posix())
new_texts = spark.createDataFrame([
    ('n1','Футболисты забили два гола и выиграли матч чемпионата.'),
    ('n2','Хоккеист забросил шайбу в овертайме.'),
    ('n3','Лыжники вышли на старт зимней эстафеты.'),
    ('n4','')],['row_id','text'])
new_fragments = (new_texts.select('row_id',F.posexplode(make_fragments('text')).alias('fragment_no','text'))
    .withColumn('label',F.lit(-1.0)).withColumn('category',F.lit('unknown')))
a = BEST_MODEL.transform(new_fragments).orderBy('row_id','fragment_no').select('prediction','probability').collect()
b = loaded.transform(new_fragments).orderBy('row_id','fragment_no').select('prediction','probability').collect()
for before,after in zip(a,b):
    assert before.prediction==after.prediction
    np.testing.assert_allclose(before.probability.toArray(),after.probability.toArray(),rtol=1e-10,atol=1e-12)
new_pred = loaded.transform(new_fragments)
feature_flags = new_pred.groupBy('row_id').agg(F.max((nnz('features')>0).cast('int')).cast('boolean').alias('has_features'))
new_objects = aggregate_objects(new_pred).join(feature_flags,'row_id').join(new_texts,'row_id')
display_new = (new_objects.withColumn('predicted_category',F.create_map(*[x for i,n in LABEL_NAMES.items()
    for x in (F.lit(float(i)),F.lit(n))])[F.col('prediction')]))
display_new.select('row_id','text','predicted_category','has_features','probability').show(truncate=100)

source_digest = hashlib.sha256()
with CSV_PATH.open('rb') as f:
    for chunk in iter(lambda:f.read(1024*1024),b''):
        source_digest.update(chunk)
config = dict(task='13-class Russian sports text classification',model=BEST_NAME,
    source='mikhailma/russian-social-media-text-classification',source_file=CSV_PATH.name,
    source_sha256=source_digest.hexdigest(),label_names=LABEL_NAMES,seed=SEED,
    split='SHA256(connected component of oid and nonempty normalized duplicate links + seed), 70/15/15 before fragmentation',
    fragment_words=FRAGMENT_WORDS,keep_short_tail=True,max_train_fragments=MAX_TRAIN_FRAGMENTS,
    vocab_size_limit=VOCAB_SIZE,fitted_vocabulary=len(cv_best.vocabulary),min_df=MIN_DF,
    reg_param=REG_PARAM,token_pattern=TOKEN_PATTERN,clean_sql=CLEAN_SQL,
    actual_model_params={p.name:str(v) for p,v in BEST_MODEL.stages[-1].extractParamMap().items()},
    test_source_metrics=TEST_SOURCE_METRICS,source_groups=SOURCE_COUNT,connected_components=COMPONENT_COUNT,
    test_fragment_metrics=TEST_FRAG_METRICS,test_object_metrics=TEST_OBJECT_METRICS,
    oov_share=OOV_SHARE,zero_vectors=ZERO_VECTORS,mean_nnz=MEAN_NNZ,
    generation=dict(category=GEN_CATEGORY,source_docs_limit=MARKOV_DOCS,max_edges=MAX_EDGES,
        sentence_pattern=r'(?<=[.!?])\s+|[\r\n]+',token_pattern=MARKOV_PATTERN,seeds=[7,42,101]),
    versions=dict(python=platform.python_version(),spark=spark.version,numpy=np.__version__,pandas=pd.__version__))
(OUT / 'model_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
test_object_predictions.select('row_id','category','label','prediction','probability').write.mode('overwrite').parquet((OUT / 'test_object_predictions').as_posix())
print('Каталог результатов:',OUT)
print('Файлов результатов:',sum(p.is_file() for p in OUT.rglob('*')))
conclusion('Полный Pipeline сохранён и загружен обратно. Классы и вероятности '
    'на четырёх фиксированных входах совпали. Правила предобработки, '
    'параметры, разбиение и таблицы генератора сохранены вместе с результатами.')

test_source_predictions.write.mode('overwrite').parquet((OUT / 'test_source_predictions').as_posix())


## 13. Итоговые выводы и ограничения

Итог ниже формируется по фактическим данным запуска. Он не содержит заранее назначенного победителя или ожидаемых значений качества.


In [ ]:
summary = f"""
1. **Данные.** Из {N_RAW:,} исходных записей после контроля и удаления полных повторов осталось {N_DOCS:,} объектов. Использованы 13 спортивных тематик. Классификатор получает только текст; идентификаторы и целевые категории не входят в признаки.
2. **Разбиение.** Нормализованные повторы объединены в {N_GROUPS:,} групп; конфликтующих групп - {CONFLICT_GROUPS:,}. Группы распределены в train/valid/test до нарезки, пересечений нет. На train ограничен вклад длинных объектов. Словарь и IDF обучены только на train.
3. **Модели.** Сравнены NB_BOW, LR_BOW, LR_TFIDF, LR_TFIDF_BIGRAM и включённые варианты стоп-слов. По macro-F1 на valid выбран {BEST_NAME}; значение по группам oid - {comparison.iloc[0].macro_f1:.4f}. Постоянное правило на valid - {baseline_valid['macro_f1']:.4f}.
4. **Итоговая оценка.** На test macro-F1 по фрагментам - {TEST_FRAG_METRICS['macro_f1']:.4f}, по исходным объектам - {TEST_OBJECT_METRICS['macro_f1']:.4f}; accuracy по объектам - {TEST_OBJECT_METRICS['accuracy']:.4f}. Macro-F1 по группам oid - {TEST_SOURCE_METRICS['macro_f1']:.4f}, постоянное правило по группам oid - {baseline_test_source['macro_f1']:.4f}. Оценены только заранее выбранная модель и baseline.
5. **Представление.** OOV - {OOV_SHARE:.2%}, нулевых фрагментов - {ZERO_VECTORS:,}, среднее nnz - {MEAN_NNZ:.1f}. Ограниченный словарь экономит память, но отбрасывает часть редкой спортивной лексики и сочетаний. Коэффициенты и ошибки позволяют исследовать такие ограничения.
6. **Генерация.** Две цепи обучены по {N_SENTENCES:,} предложениям train-корпуса {GEN_CATEGORY}. Получены шесть примеров, проверены границы, backoff, воспроизводимость и остановка. Второй порядок сохраняет более длинные локальные сочетания, но оба порядка ограничены коротким контекстом.
7. **Ограничения.** Метка описывает тематику источника и может не совпадать с темой отдельной публикации. Группы oid разделены целиком; дополнительные связи через одинаковые непустые тексты также не пересекают выборки. Смысл oid как ID именно паблика требует подтверждения источником. Близкие пересказы и репосты с изменённым текстом не полностью устраняются хеш-группировкой. Не проводились временная проверка, подбор всех гиперпараметров и независимая оценка на новых источниках. Генератор не является диалоговой или фактологической моделью.
"""
display(Markdown(summary))
(OUT / 'conclusions.md').write_text(summary,encoding='utf-8')


### Устойчивость результата

Для NB_BOW macro-F1 по группам oid снизился с **0.9693 на valid** до **0.8440 на test**. Разрыв составляет **0.1253**. Его нельзя скрывать усреднением результатов двух выборок. Группы связаны репостами и отличаются по лексике; хеш-разбиение не гарантирует одинаковую сложность частей. Наблюдаемое снижение показывает необходимость дополнительной проверки на целых независимых группах источников. Ни test, ни обнаруженные на нём ошибки не использовались для последующего изменения классификатора.


## 14. Ответы на контрольные вопросы

1. **Метки.** В учебном корпусе отзывов методички: 0 - нейтральный, 1 - положительный, 2 - отрицательный. В данной адаптации метки 0–12 заданы явно таблицей `LABEL_NAMES` и соответствуют спортивным тематикам; переносить номера классов из отзывов нельзя.
2. **Токен.** Единица текста, выделенная токенизатором. Здесь это слово, число, слово с внутрисловным дефисом/апострофом или группа `!/?`. В генераторе отдельно сохраняется основная пунктуация.
3. **Нормализация.** Нижний регистр объединяет варианты написания; удаление стоп-слов уменьшает количество служебных единиц, но может потерять отрицание. Лемматизация объединяет словоформы ценой дополнительной обработки и возможных ошибок на именах/сленге. Она не применена в основном опыте.
4. **DISPLAY_STOP.** Скрывает фоновые слова только на рисунках. Стоп-лист классификатора меняет признаки и потому сравнивается отдельно на valid.
5. **Разреженный вектор.** Хранит размерность, индексы и значения ненулевых координат. Плотная float64-матрица требует примерно `8 × N × V` байт; служебные расходы считаются отдельно.
6. **TF, DF, IDF.** TF считает вхождения, DF - документы с токеном. Spark использует `ln((N+1)/(DF+1))`. Повтор слова внутри одного документа увеличивает TF, но не DF.
7. **Обучение преобразований.** CountVectorizer и IDF обучаются по train. Фиксированные правила очистки и токенизации не оценивают статистики корпуса. Обучение словаря на test создаёт утечку.
8. **Неизвестные слова.** CountVectorizer игнорирует отсутствующие в словаре токены. HashingTF помещает любой токен в координату по хешу; возможны совпадения координат разных токенов.
9. **Масштаб и регуляризация.** Величина признака влияет на необходимый коэффициент и действие L2-штрафа. Нормирование TF–IDF меняет масштаб относительно BoW; один regParam не означает одинаковую эффективную регуляризацию разных представлений.
10. **Метрики.** Macro-F1 даёт равный вес классам, weighted-F1 взвешивает их по support. Precision - диагональная ячейка, делённая на сумму столбца; recall - на сумму строки.
11. **Разность коэффициентов.** Для softmax разность весов двух классов входит в логарифм отношения их вероятностей. Это условная ассоциация внутри выбранной модели и масштаба признаков.
12. **OOV и nnz.** OOV - доля вхождений вне обученного словаря; nnz - число ненулевых координат готового вектора. Для нулевого вектора проверяют пустой текст, алфавит, правила очистки, редкие слова, стоп-лист и нулевые IDF.
13. **Переходы.** Первому порядку нужны биграммы, второму - триграммы. Маркеры BOS/EOS и раздельная обработка предложений запрещают переходы через их границы.
14. **Seed, температура, backoff.** Seed фиксирует случайный выбор при одинаковом порядке вариантов. Температура меняет концентрацию распределения. При неизвестной паре выполняется поиск по одному слову, затем по стартовому состоянию.
15. **Порядок разбиения.** Сначала целые исходные объекты или группы произведений делят на train/valid/test; затем нарезают фрагменты. Обратный порядок позволил бы соседним частям одного объекта оказаться и в обучении, и в тесте.
16. **Интерпретация авторства и темы.** Имена персонажей, названия клубов, сюжеты и оформление способны стать признаками источника вместо устойчивого стиля или темы. Для переноса нужны проверки на целиком новых произведениях/сообществах и анализ содержимого признаков.
17. **Прогноз целому объекту.** Вероятности его фрагментов усредняются покомпонентно, затем выбирается максимальная компонента. Метрики по объектам дают каждому объекту один вес. Без ID сообщества это не оценка по пабликам.
18. **Распределение вычислений.** Spark читает и преобразует корпус, считает частоты, обучает модели, агрегирует матрицы ошибок и переходы. Python строит графики по малым таблицам и последовательно генерирует короткие тексты по ограниченной таблице переходов.

**Источники:**

- Приложенная методичка «Практическая работа №6. Обработка текстовых данных с использованием Spark MLlib», 43 страницы.
- Приложенный ноутбук-пример `Pract5_Spark_ML.ipynb` - структура изложения и оформления.
- [Датасет Kaggle](https://www.kaggle.com/datasets/mikhailma/russian-social-media-text-classification/data).
- [Spark: извлечение и преобразование признаков](https://spark.apache.org/docs/3.5.3/ml-features.html).
- [Spark: классификация и регрессия](https://spark.apache.org/docs/3.5.3/ml-classification-regression.html).
- [Spark: Pipeline](https://spark.apache.org/docs/3.5.3/ml-pipeline.html).


In [ ]:
# Выполнять после завершения работы с моделями в текущем ядре.
spark.catalog.clearCache()
spark.stop()
print('Spark остановлен. Результаты сохранены в', OUT)
